# 05 · Analysis: input checks before adding features
Three checks; results go to `results/analysis_inputs.txt` next to this notebook (**commit & push it**).

| Part | Question |
|---|---|
| A · Fake zeros | Does our 1-min telemetry have Phase 1's "polling artifact" (daytime 0 MW at some minutes)? Does it drag our 15-min target down? |
| B · Telemetry delay | How late does telemetry arrive (`Load_Datetime`)? Decides whether "actual output up to the forecast time" features are realistic |
| C · Reuniwatt vs Solcast | Variables in each; agreement with the plant sensors; correlation with the target; **does Solcast history behave like forecasts or like actual weather?** |

Naming: **pv1 = small plant (~935 MW)**, **pv2 = large plant (~1,584 MW)** (Reuniwatt / sensor naming).

In [0]:
dbutils.widgets.text("output_schema", "ewec_dev_powerops.scratch_yash", "Output schema (pipeline tables)")
dbutils.widgets.text("overlap_start", "2024-01-23", "Reuniwatt/Solcast overlap start")
dbutils.widgets.text("overlap_end", "2025-11-18", "Overlap end (Solcast stops 2025-11-19)")

In [0]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(REPO, "src"))
for m in [m for m in list(sys.modules) if m.startswith("uc2_solar")]:
    del sys.modules[m]
from datetime import datetime
from functools import reduce
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from uc2_solar.spark_utils import add_solar, ceil_to, gulf_to_utc
from uc2_solar.pipelines.data_engineering.nodes import _clean_mw, _detect_swap_date

spark.conf.set("spark.sql.session.timeZone", "UTC")
S = dbutils.widgets.get("output_schema")
O0, O1 = dbutils.widgets.get("overlap_start"), dbutils.widgets.get("overlap_end")
DE = {"pv_sentinel_abs_mw": 5000, "night_neg_floor_mw": -10, "swap_date": "auto"}
SITES = {"pv1": {"capacity_mw": 935, "lat": 24.45, "lon": 54.6}, "pv2": {"capacity_mw": 1584, "lat": 24.45, "lon": 54.6}}
PV = spark.table("ewec_dev_powerops.bronze.power_contango_minute_pv").withColumn("ts_gulf", F.col("DateTime").cast("timestamp"))
SWAP = _detect_swap_date(PV, DE)
swap = F.lit(SWAP)
small = F.when(F.to_date("ts_gulf") < swap, _clean_mw("PV2_MW", DE)).otherwise(_clean_mw("PV1_MW", DE))
large = F.when(F.to_date("ts_gulf") < swap, _clean_mw("PV1_MW", DE)).otherwise(_clean_mw("PV2_MW", DE))
REPORT = [f"# Input analysis · run {datetime.utcnow():%Y-%m-%d %H:%M} UTC · Bronze MW column swap {SWAP}"]


def as_text(df, title, max_rows=200):
    rows = df.limit(max_rows).collect()
    f = lambda v: "" if v is None else (f"{v:.4g}" if isinstance(v, float) else str(v))
    return "\n".join([f"## {title}", " | ".join(df.columns)] + [" | ".join(f(v) for v in r) for r in rows]) + "\n"


def show(df, title, max_rows=200):
    display(df)
    REPORT.append(as_text(df, title, max_rows))

## A · Fake zeros (polling artifact)
**A1:** share of daytime minutes (09:00–15:00 Gulf) at exactly 0 MW, by minute of the hour. A real outage hits all minutes equally; a polling artifact shows **spikes at particular minutes**.

In [0]:
day = PV.where(F.hour("ts_gulf").between(9, 14)).select("ts_gulf", small.alias("small"), large.alias("large"))
show(day.groupBy(F.minute("ts_gulf").alias("minute")).agg(
    F.count("*").alias("minutes"),
    F.round(100 * F.avg((F.col("small") == 0).cast("int")), 2).alias("pv1_small_zero_pct"),
    F.round(100 * F.avg((F.col("large") == 0).cast("int")), 2).alias("pv2_large_zero_pct")).orderBy("minute"),
    "A1 · daytime zero share by minute of hour (09-15 Gulf)")

**A2:** 15-min periods (daytime) that mix zero minutes with real output (> 100 MW), i.e. where a fake zero would pull the 15-min average down, by year.

In [0]:
b = (day.withColumn("p15", ceil_to("ts_gulf", 15))
     .groupBy("p15").agg(*[F.sum((F.col(c) == 0).cast("int")).alias(f"{c}_zero_min") for c in ("small", "large")],
                         *[F.max(c).alias(f"{c}_max") for c in ("small", "large")], F.count("*").alias("n")))
show(b.groupBy(F.year("p15").alias("year")).agg(
    F.count("*").alias("periods"),
    F.round(100 * F.avg(((F.col("small_zero_min") > 0) & (F.col("small_max") > 100)).cast("int")), 2).alias("pv1_mixed_pct"),
    F.round(100 * F.avg(((F.col("large_zero_min") > 0) & (F.col("large_max") > 100)).cast("int")), 2).alias("pv2_mixed_pct")).orderBy("year"),
    "A2 · daytime 15-min periods mixing zero minutes with output > 100 MW")

✅ **Read:** flat A1 and **mixed % ≈ 0** → no artifact, no fix needed. Spikes at certain minutes and mixed % of several % → apply a fix (treat daytime zeros inside producing periods as missing).

## B · Telemetry delay (`Load_Datetime`)
Delay = load time − measurement time, in minutes. `DateTime` is Gulf time; if `Load_Datetime` is UTC, every delay is shifted by −240 min (both shown).

In [0]:
ld = F.coalesce(*[F.try_to_timestamp(F.col("Load_Datetime"), F.lit(fmt)) for fmt in
                  ["yyyy-MM-dd HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss", "yyyy-MM-dd HH:mm:ss.SSS", "dd/MM/yyyy HH:mm:ss", "MM/dd/yyyy HH:mm:ss"]])
dl = PV.where(F.col("ts_gulf") >= F.add_months(F.current_date(), -2)).withColumn("load_ts", ld) \
    .withColumn("delay_min", (F.unix_timestamp("load_ts") - F.unix_timestamp("ts_gulf")) / 60)
display(dl.select("DateTime", "Load_Datetime", "load_ts", "delay_min").orderBy(F.desc("DateTime")).limit(10))
show(dl.groupBy(F.date_trunc("week", "ts_gulf").alias("week")).agg(
    F.round(100 * F.avg(F.col("load_ts").isNull().cast("int")), 1).alias("unparsed_pct"),
    F.percentile_approx("delay_min", [0.05, 0.5, 0.95]).alias("delay_p5_p50_p95_as_is"),
    F.percentile_approx(F.col("delay_min") + 240, [0.05, 0.5, 0.95]).alias("delay_p5_p50_p95_if_load_utc"),
    F.countDistinct("load_ts").alias("distinct_load_times")).orderBy("week"), "B · telemetry delay (last 2 months)")

✅ **Read:** the median delay (in whichever column looks plausible, i.e. not negative) tells us how fresh data is at forecast time. A few minutes → "actual up to the origin" features are realistic. Hours or once-a-day loads → only use data up to the previous day.

## C · Reuniwatt vs Solcast vs plant sensors (overlap period)
Everything on 15-min periods (UTC, period end), daytime only (clear-sky GHI > 50).
- **Solcast:** Bronze `pv1/pv2_history_solcast` (which table belongs to which plant is checked in C2).
- **Reuniwatt:** pipeline table `uc2_int_reuniwatt_5min` (day-ahead), as the **latest** run per period and as a **12–24 h ahead** run.
- **Sensors and target:** Bronze telemetry.

In [0]:
def to15(df, ts, cols, by=("site",)):
    return df.withColumn("period_end", ceil_to(ts, 15)).groupBy(*by, "period_end").agg(*[F.avg(c).alias(c) for c in cols])

in_overlap = F.to_date("period_end").between(F.lit(O0), F.lit(O1))
# Sensors + target
sens_long = reduce(DataFrame.unionByName, [
    PV.select(F.lit("pv1").alias("site"), gulf_to_utc("ts_gulf").alias("ts"), small.alias("actual_mw"),
              F.col("PV1_GHI").alias("m_ghi"), ((F.col("PV1_ArrayRadiance_east") + F.col("PV1_ArrayRadiance_west")) / 2).alias("m_gti"),
              F.col("PV1_Temp").alias("m_temp"), F.col("PV1_HumidityPer").alias("m_rh"), F.lit(None).cast("double").alias("m_wind")),
    PV.select(F.lit("pv2").alias("site"), gulf_to_utc("ts_gulf").alias("ts"), large.alias("actual_mw"),
              F.col("PV2_GHI").alias("m_ghi"), F.col("PV2_ArrayRadiance").alias("m_gti"),
              F.col("PV2_Outdoor_Temp").alias("m_temp"), F.col("PV2_HumidityPer").alias("m_rh"), F.col("PV2_WindSpeed").alias("m_wind")),
])
sens = to15(sens_long, "ts", ["actual_mw", "m_ghi", "m_gti", "m_temp", "m_rh", "m_wind"]).where(in_overlap)
sens = add_solar(sens, SITES, "period_end", 15).where("clearsky_ghi > 50")

# Solcast (both tables, for both plants, so the mapping can be checked)
SC_COLS = {"ghi": "ghi", "dni": "dni", "dhi": "dhi", "gti": "gti", "air_temp": "temp", "relative_humidity": "rh",
           "wind_speed_10m": "wind", "cloud_opacity": "cloud"}
sc_tables = {}
for t in ["pv1", "pv2"]:
    raw = spark.table(f"ewec_dev_powerops.bronze.{t}_history_solcast")
    REPORT.append(f"Solcast {t} table columns: {', '.join(raw.columns)}")
    have = {k: v for k, v in SC_COLS.items() if k in raw.columns}
    sc = raw.select(F.col("period_end").cast("timestamp").alias("ts"), *[F.col(k).cast("double").alias(f"sc_{v}") for k, v in have.items()])
    sc_tables[t] = to15(sc.withColumn("site", F.lit("x")), "ts", [f"sc_{v}" for v in have.values()]).drop("site").where(in_overlap)

# Reuniwatt day-ahead: latest run, and 12-24 h ahead
ru = spark.table(f"{S}.uc2_int_reuniwatt_5min").where("horizon = 'dayahead'").where(F.to_date("period_end").between(F.lit(O0), F.lit(O1)))
REPORT.append(f"Reuniwatt columns: {', '.join(ru.columns)}")
RU = {"ghi": "ghi", "dni": "dni", "dhi": "dhi", "gti_poa": "gti", "temp_c": "temp", "humidity_pct": "rh", "wind_speed": "wind",
      "cloud_opacity": "cloud", "power_mw": "power"}
latest = (ru.withColumn("_rk", F.row_number().over(Window.partitionBy("site", "period_end").orderBy("lead_min"))).where("_rk = 1"))
ru_latest = to15(latest.select("site", "period_end", *[F.col(k).alias(f"rl_{v}") for k, v in RU.items()]), "period_end", [f"rl_{v}" for v in RU.values()])
far = ru.where("lead_min > 720 AND lead_min <= 1440")
ru_far = to15(far.select("site", "period_end", *[F.col(k).alias(f"rf_{v}") for k, v in RU.items()]), "period_end", [f"rf_{v}" for v in RU.values()])
base = sens.join(ru_latest, ["site", "period_end"], "left").join(ru_far, ["site", "period_end"], "left")
for t, df in sc_tables.items():
    base = base.join(df.select("period_end", *[F.col(c).alias(c.replace("sc_", f"sc{t[-1]}_")) for c in df.columns if c.startswith("sc_")]), "period_end", "left")
tbl = f"{S}.uc2_analysis_inputs_15min"
base.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl)
base = spark.table(tbl)
print("rows per site:", {r[0]: r[1] for r in base.groupBy("site").count().collect()})

**C1 · Which variables each source has** (printed into the report above: column lists). **C2 · Which Solcast table is which plant:** correlation of each table's GTI and GHI with each plant's actual output.

In [0]:
c2 = reduce(DataFrame.unionByName, [
    base.groupBy("site").agg(*[F.round(F.corr(f"sc{k}_{v}", "actual_mw"), 4).alias(f"corr_{v}_vs_actual") for v in ("ghi", "gti")],
                             *[F.round(F.corr(f"sc{k}_ghi", "m_ghi"), 4).alias("corr_ghi_vs_sensor")]).withColumn("solcast_table", F.lit(f"pv{k}"))
    for k in ("1", "2")])
show(c2.orderBy("site", "solcast_table"), "C2 · Solcast table vs plant (correlations)")

**C3 · Agreement with plant sensors.** For each plant, the **matching** Solcast table (same label), Reuniwatt latest and Reuniwatt 12–24 h ahead vs the sensor: correlation, bias and error (% of mean measured).

In [0]:
def agree(df, f, m, label):
    d = df.where(F.col(f).isNotNull() & F.col(m).isNotNull())
    return d.groupBy("site").agg(F.count("*").alias("n"), F.round(F.corr(f, m), 4).alias("corr"),
                                 F.round(F.avg(F.col(f) - F.col(m)), 2).alias("bias"),
                                 F.round(100 * F.avg(F.abs(F.col(f) - F.col(m))) / F.avg(F.abs(m)), 2).alias("err_pct_of_mean")).withColumn("source", F.lit(label))

# Matching Solcast table per plant (same label). A variable missing from a table (e.g. humidity for pv2) stays blank.
NULLD = F.lit(None).cast("double")
matched = base
for v in ["ghi", "gti", "temp", "rh", "wind"]:
    c1 = F.col(f"sc1_{v}") if f"sc1_{v}" in base.columns else NULLD
    c2 = F.col(f"sc2_{v}") if f"sc2_{v}" in base.columns else NULLD
    matched = matched.withColumn(f"sc_{v}", F.when(F.col("site") == "pv1", c1).otherwise(c2))
print("Solcast columns available:", sorted(c for c in base.columns if c.startswith("sc")))
rows = []
for var, sensor in [("ghi", "m_ghi"), ("gti", "m_gti"), ("temp", "m_temp"), ("rh", "m_rh"), ("wind", "m_wind")]:
    for src, pre in [("Solcast history", "sc"), ("Reuniwatt latest", "rl"), ("Reuniwatt 12-24h", "rf")]:
        rows.append(agree(matched, f"{pre}_{var}", sensor, src).withColumn("variable", F.lit(var)))
c3 = reduce(DataFrame.unionByName, rows).select("variable", "site", "source", "n", "corr", "bias", "err_pct_of_mean")
show(c3.orderBy("variable", "site", "source"), "C3 · agreement with plant sensors (daytime)")

**C4 · Correlation with the target (actual MW, daytime).** Sensors = ceiling (perfect knowledge of the weather).

In [0]:
inputs = {"sensor GHI": "m_ghi", "sensor GTI": "m_gti", "Solcast GHI": "sc_ghi", "Solcast GTI": "sc_gti",
          "Reuniwatt latest GHI": "rl_ghi", "Reuniwatt latest GTI": "rl_gti", "Reuniwatt latest power": "rl_power",
          "Reuniwatt 12-24h GHI": "rf_ghi", "Reuniwatt 12-24h GTI": "rf_gti", "Reuniwatt 12-24h power": "rf_power"}
c4 = reduce(DataFrame.unionByName, [
    matched.groupBy("site").agg(F.count(F.when(F.col(c).isNotNull(), 1)).alias("n"), F.round(F.corr(c, "actual_mw"), 4).alias("corr_with_actual")).withColumn("input", F.lit(k))
    for k, c in inputs.items()])
show(c4.orderBy("site", F.desc("corr_with_actual")), "C4 · correlation with actual output (daytime)")

**C5 · By month: Solcast vs sensor and Reuniwatt vs sensor (GHI error % of mean).** If Solcast is consistently far closer to the sensors than even Reuniwatt's latest run, Solcast history behaves like **actual weather**, not forecasts.

In [0]:
c5 = matched.groupBy("site", F.date_trunc("month", "period_end").alias("month")).agg(
    *[F.round(100 * F.avg(F.abs(F.col(f"{p}_ghi") - F.col("m_ghi"))) / F.avg("m_ghi"), 2).alias(f"{n}_ghi_err_pct")
      for p, n in [("sc", "solcast"), ("rl", "reuni_latest"), ("rf", "reuni_12_24h")]],
    *[F.round(F.corr(f"{p}_ghi", "m_ghi"), 4).alias(f"{n}_ghi_corr") for p, n in [("sc", "solcast"), ("rl", "reuni_latest")]])
show(c5.orderBy("site", "month"), "C5 · GHI error vs sensors by month")

ratio = c3.where("variable = 'ghi'").groupBy("site").pivot("source").agg(F.first("err_pct_of_mean")).collect()
for r in ratio:
    d = r.asDict()
    s, rl = d.get("Solcast history"), d.get("Reuniwatt latest")
    if s is not None and rl:
        verdict = ("Solcast history is much closer to the sensors than any Reuniwatt forecast -> behaves like ACTUAL weather (Phase 1 backtest likely optimistic)"
                   if s < 0.6 * rl else "Solcast history is about as close to the sensors as a forecast -> behaves like a FORECAST")
        REPORT.append(f"VERDICT {d['site']}: Solcast GHI error {s}% vs Reuniwatt latest {rl}% -> {verdict}")
print("\n".join(REPORT[-2:]))

## Write the report

In [0]:
text = "\n".join(REPORT)
os.makedirs("results", exist_ok=True)
with open("results/analysis_inputs.txt", "w") as fh:
    fh.write(text)
print(f"Saved {os.path.abspath('results/analysis_inputs.txt')} — commit & push it.\n")
print(text)